# Challenge MercadoLibre — VIS Practicantes

**Fuente de datos:** World Bank Development Indicators (actualización: 2023-03-01)  
**Indicadores analizados:** % personas con internet · Suscripciones celular · Población total  
**Período:** 2000–2021 · **Países:** 217 (excluye agregados regionales)  
**Motor SQL:** DuckDB (in-process)  

---

## Estructura del análisis

| Sección | Contenido |
|---|---|
| 1 | Setup e importaciones |
| 2 | Carga y limpieza de datos (Wide → Long) |
| 3 | Carga a DuckDB |
| 4 | Query 1 — Indicadores por Año y País |
| 5 | Query 2 — Crecimiento YoY desde 2010 |
| 6 | Visualizaciones |
| 7 | Conclusiones y relación con MercadoLibre |


## Sección 1 — Setup e importaciones

In [1]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Rutas de archivos (relativas a la carpeta notebooks/)
FILE_CELULAR  = '../data/BASE_CASO_PRATICA_VIS_20230424__1___1___1___2_.xls'
FILE_INTERNET = '../data/BASE_CASO_PRATICA_VIS_20230424_2__1___1___1___2_.xlsx'
FILE_POBLACION = '../data/BASE_CASO_PRATICA_VIS_20230424_3__1___1___1___2_.xls'

print('Librerías cargadas correctamente ✓')
print(f'  DuckDB  : {duckdb.__version__}')
print(f'  Pandas  : {pd.__version__}')


Librerías cargadas correctamente ✓
  DuckDB  : 1.5.4
  Pandas  : 3.0.3


## Sección 2 — Carga y limpieza de datos

### Decisiones de calidad de datos

1. **Formato Wide → Long:** los archivos del World Bank tienen una columna por año. Para SQL necesitamos una fila por (país, año). Se aplica `melt()`.
2. **Agregados regionales:** 49 códigos como `AFE`, `ARB`, `EUU` son sumas regionales, no países. Se excluyen usando la hoja `Metadata - Countries` (campo `Income_Group = 'Agregados'`).
3. **NULLs en 2021:** el 72 % de los países no tiene dato de internet en 2021 porque el World Bank aún no lo había publicado al momento de la extracción. Se conservan como NULL, no se imputan.
4. **Archivo de población (File 3):** contiene datos desde 1960. Solo se usa 2000–2021 para alinear con los otros indicadores.


In [ ]:
def leer_world_bank(path, engine=None):
    """
    Lee un archivo World Bank en formato wide.
    Salta las primeras 3 filas de metadatos del Banco Mundial
    y retorna un DataFrame limpio.
    """
    kw = {'skiprows': 3, 'header': 0, 'sheet_name': 'Data'}
    if engine:
        kw['engine'] = engine
    df = pd.read_excel(path, **kw)
    return df.dropna(how='all')  # elimina filas completamente vacías


def wide_a_long(df, nombre_valor):
    """
    Convierte formato wide (columnas = años) a long (filas = observaciones).
    Cada fila resultante representa: un país en un año con su valor del indicador.
    """
    cols_id   = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code']
    cols_anio = [c for c in df.columns if str(c).isdigit()]

    long = df[cols_id + cols_anio].melt(
        id_vars    = cols_id,
        var_name   = 'year',
        value_name = nombre_valor
    )
    long['year'] = long['year'].astype(int)

    return long.rename(columns={
        'Country Name'   : 'country_name',
        'Country Code'   : 'country_code',
        'Indicator Name' : 'indicator_name',
        'Indicator Code' : 'indicator_code',
    })


# ── Leer los 3 archivos ────────────────────────────────────────────────
raw_celular   = leer_world_bank(FILE_CELULAR,   engine='xlrd')
raw_internet  = leer_world_bank(FILE_INTERNET)
raw_poblacion = leer_world_bank(FILE_POBLACION, engine='xlrd')

# ── Leer metadatos de países (para filtrar agregados y obtener región) ──
meta = pd.read_excel(FILE_CELULAR, engine='xlrd', sheet_name='Metadata - Countries')
meta = meta.rename(columns={
    'Country Name' : 'country_name',
    'Country Code' : 'country_code',
    'Region'       : 'region',
    'Income_Group' : 'income_group',
})

print(f'File 1 (celular)   : {raw_celular.shape[0]} países, {raw_celular.shape[1]-4} años')
print(f'File 2 (internet)  : {raw_internet.shape[0]} países, {raw_internet.shape[1]-4} años')
print(f'File 3 (población) : {raw_poblacion.shape[0]} países, {raw_poblacion.shape[1]-4} años')
print(f'Metadata           : {len(meta)} registros')


In [ ]:
# ── Convertir a formato long ───────────────────────────────────────────
long_celular   = wide_a_long(raw_celular,   'mobile_subscriptions')
long_internet  = wide_a_long(raw_internet,  'internet_pct')
long_poblacion = wide_a_long(raw_poblacion, 'population')

# ── Unir los 3 indicadores en una sola tabla ───────────────────────────
# Base: internet (2000-2021). Se hace LEFT JOIN para no perder países
# que tengan internet pero les falte algún dato de celular o población.
world_indicators = (
    long_internet[['country_name','country_code','year','internet_pct']]
    .merge(long_celular[['country_code','year','mobile_subscriptions']],
           on=['country_code','year'], how='left')
    .merge(long_poblacion[['country_code','year','population']],
           on=['country_code','year'], how='left')
)

# ── Resumen de calidad ─────────────────────────────────────────────────
agregados = meta[meta['income_group']=='Agregados']['country_code'].tolist()

print(f'Tabla unificada (world_indicators):')
print(f'  Total filas       : {len(world_indicators):,}')
print(f'  Países totales    : {world_indicators["country_code"].nunique()}')
print(f'  Agregados a excl. : {len(agregados)}')
print(f'  Países reales     : {world_indicators["country_code"].nunique() - len(agregados)}')
print(f'  Rango de años     : {world_indicators["year"].min()} – {world_indicators["year"].max()}')
print()
print('NULLs por indicador:')
print(world_indicators[['internet_pct','mobile_subscriptions','population']].isna().sum())


## Sección 3 — Carga a DuckDB

DuckDB permite ejecutar SQL directamente sobre DataFrames de pandas sin necesidad de un servidor.
Se registran las dos tablas como vistas en memoria.


In [ ]:
# Crear conexión en memoria
con = duckdb.connect()

# Registrar DataFrames como tablas SQL
con.register('world_indicators',   world_indicators)
con.register('countries_metadata', meta)

# Verificar tablas disponibles
tablas = con.execute("SHOW TABLES").df()
print('Tablas registradas en DuckDB:')
print(tablas)

# Vista previa de world_indicators
print('\nVista previa — world_indicators:')
con.execute("""
    SELECT *
    FROM world_indicators
    WHERE country_code = 'COL'
    LIMIT 5
""").df()


## Sección 4 — Query 1: Indicadores por Año y País

**Objetivo:** tabla con los tres indicadores para cada combinación de año y país.

**Decisiones de diseño:**
- `INNER JOIN` con `countries_metadata` para poder filtrar agregados regionales y enriquecer con región e income group.
- `ROUND(internet_pct, 2)` para presentar el porcentaje con precisión legible.
- `WHERE income_group != 'Agregados'` excluye los 49 códigos regionales del World Bank.
- Orden: año ascendente, luego país alfabético (facilita lectura y validación).


In [ ]:
SQL_QUERY_1 = """
-- ================================================================
-- QUERY 1: Indicadores por Año y País
-- Fuente  : World Bank Development Indicators (2000-2021)
-- Excluye : agregados regionales (income_group = 'Agregados')
-- Nota    : NULLs en 2021 para internet_pct son esperados —
--           el World Bank no había publicado esos datos al momento
--           de la extracción (2023-03-01)
-- ================================================================

SELECT
    t.year                              AS anio,
    t.country_name                      AS pais,
    t.country_code                      AS codigo_pais,
    m.region,
    m.income_group,
    ROUND(t.internet_pct, 2)            AS pct_personas_internet,
    t.mobile_subscriptions              AS suscripciones_celular,
    t.population                        AS poblacion_total

FROM world_indicators t
INNER JOIN countries_metadata m
    ON t.country_code = m.country_code

WHERE m.income_group != 'Agregados'

ORDER BY
    t.year        ASC,
    t.country_name ASC
"""

resultado_q1 = con.execute(SQL_QUERY_1).df()

print(f'Query 1 — filas: {len(resultado_q1):,} | países únicos: {resultado_q1["codigo_pais"].nunique()}')
resultado_q1.head(10)


In [ ]:
# ── Muestra LATAM año 2020 ─────────────────────────────────────────────
LATAM = ['ARG','BOL','BRA','CHL','COL','CRI','CUB','DOM','ECU',
         'GTM','HND','MEX','NIC','PAN','PER','PRY','SLV','URY','VEN']

latam_2020 = resultado_q1[
    (resultado_q1['anio'] == 2020) &
    (resultado_q1['codigo_pais'].isin(LATAM))
].sort_values('pct_personas_internet', ascending=False)

print('LATAM — año 2020 (ordenado por % internet)')
latam_2020[['pais','anio','pct_personas_internet','suscripciones_celular','poblacion_total']]


In [ ]:
# ── Exportar resultado completo ────────────────────────────────────────
resultado_q1.to_csv('../outputs/query1_indicadores_anio_pais.csv', index=False)
print('Archivo guardado: outputs/query1_indicadores_anio_pais.csv ✓')


## Sección 5 — Query 2: Crecimiento YoY desde 2010

**Objetivo:** calcular el crecimiento porcentual año contra año (YoY) de los tres indicadores para cada país, desde 2010.

**Conceptos clave:**
- **`LAG(valor) OVER (PARTITION BY country_code ORDER BY year)`**: función de ventana que devuelve el valor del año anterior *dentro del mismo país*. Sin el `PARTITION BY`, tomaría el año anterior del dataset global (otro país).
- **`NULLIF(denominador, 0)`**: evita división por cero. Si el año anterior era 0, devuelve NULL en lugar de error.
- **CTE (`WITH base AS`)**: separa la lógica en dos pasos. Primero se calculan los LAGs; luego se aplica el filtro de año y la fórmula de crecimiento. Más legible que subqueries anidados.
- **NULL en 2010**: es semánticamente correcto. El challenge pide datos desde 2010, por lo que no existe un año 2009 para calcular el crecimiento inicial. No es un error de datos.
- **"No pueden faltar países"**: todos los países aparecen aunque tengan NULLs en algunos años, porque el filtro `year >= 2010` actúa sobre la tabla ya procesada con los LAGs.


In [ ]:
SQL_QUERY_2 = """
-- ================================================================
-- QUERY 2: Crecimiento YoY (Year-over-Year) desde 2010
-- Fórmula : (valor_actual - valor_anterior) / valor_anterior * 100
-- LAG()   : obtiene el valor del año anterior por país
--           PARTITION BY country_code → el LAG no cruza países
-- NULL 2010: sin año anterior disponible → YoY indefinido (correcto)
-- ================================================================

WITH base AS (
    SELECT
        t.country_code,
        t.country_name,
        m.region,
        m.income_group,
        t.year,

        -- Valores del año actual
        t.internet_pct,
        t.mobile_subscriptions,
        t.population,

        -- Valores del año anterior (por país, no global)
        LAG(t.internet_pct)         OVER (PARTITION BY t.country_code ORDER BY t.year) AS internet_pct_prev,
        LAG(t.mobile_subscriptions) OVER (PARTITION BY t.country_code ORDER BY t.year) AS mobile_prev,
        LAG(t.population)           OVER (PARTITION BY t.country_code ORDER BY t.year) AS population_prev

    FROM world_indicators t
    INNER JOIN countries_metadata m
        ON t.country_code = m.country_code

    WHERE m.income_group != 'Agregados'
)

SELECT
    country_name                                  AS pais,
    country_code                                  AS codigo_pais,
    region,
    year                                          AS anio,

    -- Valores actuales
    ROUND(internet_pct, 2)                        AS pct_internet,
    mobile_subscriptions                          AS suscripciones_celular,
    population                                    AS poblacion,

    -- YoY — NULL en 2010 es correcto (no existe 2009 en el dataset)
    ROUND((internet_pct - internet_pct_prev)
          / NULLIF(internet_pct_prev, 0) * 100, 2)    AS yoy_internet_pct,

    ROUND((mobile_subscriptions - mobile_prev)
          / NULLIF(mobile_prev, 0) * 100, 2)           AS yoy_celular_pct,

    ROUND((population - population_prev)
          / NULLIF(population_prev, 0) * 100, 2)       AS yoy_poblacion_pct

FROM base
WHERE year >= 2010
ORDER BY country_name, year
"""

resultado_q2 = con.execute(SQL_QUERY_2).df()

print(f'Query 2 — filas: {len(resultado_q2):,} | países únicos: {resultado_q2["codigo_pais"].nunique()}')
resultado_q2.head(10)


In [ ]:
# ── Muestra Colombia ───────────────────────────────────────────────────
print('Colombia — crecimiento YoY 2010-2021')
cols_show = ['pais','anio','pct_internet','yoy_internet_pct','yoy_celular_pct','yoy_poblacion_pct']
resultado_q2[resultado_q2['codigo_pais']=='COL'][cols_show]


In [ ]:
# ── Exportar resultado completo ────────────────────────────────────────
resultado_q2.to_csv('../outputs/query2_crecimiento_yoy.csv', index=False)
print('Archivo guardado: outputs/query2_crecimiento_yoy.csv ✓')


## Sección 6 — Visualizaciones

Tres gráficas que soportan las conclusiones del Punto 3.


In [ ]:
# Paleta y países a destacar
HIGHLIGHT = ['BRA','MEX','COL','ARG','CHL']
COLORES   = {
    'BRA': '#FFB300',
    'MEX': '#E53935',
    'COL': '#1E88E5',
    'ARG': '#43A047',
    'CHL': '#8E24AA',
}

latam_q2 = resultado_q2[resultado_q2['codigo_pais'].isin(LATAM)].copy()


In [ ]:
# ────────────────────────────────────────────────────────────────────────
# GRÁFICA 1 — Evolución % internet LATAM (2010-2021)
# Muestra qué países lideran penetración y cuáles tienen espacio para crecer
# ────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('#FAFAFA')
ax.set_facecolor('#FAFAFA')

for code in LATAM:
    df_c = latam_q2[latam_q2['codigo_pais'] == code].dropna(subset=['pct_internet'])
    if df_c.empty:
        continue
    if code in HIGHLIGHT:
        ax.plot(df_c['anio'], df_c['pct_internet'],
                color=COLORES[code], linewidth=2.4, zorder=3,
                label=df_c['pais'].iloc[0])
        ax.annotate(df_c['pais'].iloc[0],
                    xy=(df_c['anio'].iloc[-1], df_c['pct_internet'].iloc[-1]),
                    xytext=(4, 0), textcoords='offset points',
                    fontsize=8.5, color=COLORES[code], va='center', fontweight='bold')
    else:
        ax.plot(df_c['anio'], df_c['pct_internet'],
                color='#BDBDBD', linewidth=0.9, zorder=1, alpha=0.6)

ax.set_title('% de personas con internet — LATAM (2010–2021)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Año', fontsize=10)
ax.set_ylabel('% población con internet', fontsize=10)
ax.set_xlim(2010, 2021)
ax.set_ylim(0, 100)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines[['top','right']].set_visible(False)
ax.legend(loc='upper left', fontsize=8.5, framealpha=0)
plt.tight_layout()
plt.savefig('../outputs/grafica1_internet_latam.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ────────────────────────────────────────────────────────────────────────
# GRÁFICA 2 — Heatmap YoY crecimiento en internet por país (2011-2021)
# Permite identificar mercados en 'ventana de aceleración'
# ────────────────────────────────────────────────────────────────────────
pivot = latam_q2[latam_q2['anio'] >= 2011].pivot_table(
    index='pais', columns='anio', values='yoy_internet_pct'
)
# Ordenar por promedio de crecimiento (más dinámicos arriba)
pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(13, 8))
fig.patch.set_facecolor('#FAFAFA')

sns.heatmap(
    pivot, annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.4, linecolor='#EEEEEE',
    cbar_kws={'label': 'Crecimiento YoY (%)', 'shrink': 0.6},
    ax=ax, annot_kws={'size': 7}
)
ax.set_title('Crecimiento YoY (%) en penetración de internet — LATAM (2011–2021)',
             fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Año', fontsize=10)
ax.set_ylabel('')
ax.tick_params(axis='y', labelsize=8.5)
ax.tick_params(axis='x', labelsize=9)
plt.tight_layout()
plt.savefig('../outputs/grafica2_yoy_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ────────────────────────────────────────────────────────────────────────
# GRÁFICA 3 — Bubble chart: Internet vs Celular, tamaño = población (2020)
# Muestra la oportunidad de mercado: eje X = penetración digital,
# eje Y = masa de suscriptores celulares, burbuja = tamaño del mercado
# ────────────────────────────────────────────────────────────────────────
latam_2020_q1 = resultado_q1[
    (resultado_q1['anio'] == 2020) &
    (resultado_q1['codigo_pais'].isin(LATAM))
].dropna(subset=['pct_personas_internet','poblacion_total'])

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#FAFAFA')
ax.set_facecolor('#FAFAFA')

colores_burbuja = [COLORES.get(c, '#90CAF9') for c in latam_2020_q1['codigo_pais']]
sizes = (latam_2020_q1['poblacion_total'] / latam_2020_q1['poblacion_total'].max() * 2800) + 80

ax.scatter(
    latam_2020_q1['pct_personas_internet'],
    latam_2020_q1['suscripciones_celular'] / 1e6,
    s=sizes, c=colores_burbuja, alpha=0.75,
    edgecolors='white', linewidth=1.2, zorder=3
)

for _, row in latam_2020_q1.iterrows():
    ax.annotate(
        row['codigo_pais'],
        xy=(row['pct_personas_internet'], row['suscripciones_celular'] / 1e6),
        ha='center', va='center',
        fontsize=7.5, fontweight='bold', color='white', zorder=4
    )

ax.set_title(
    'Oportunidad de mercado LATAM\nInternet vs Suscripciones celular (2020) — tamaño = población',
    fontsize=11, fontweight='bold', pad=12
)
ax.set_xlabel('% población con internet', fontsize=10)
ax.set_ylabel('Suscripciones celular (millones)', fontsize=10)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.grid(linestyle='--', alpha=0.35)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/grafica3_bubble_mercado.png', dpi=150, bbox_inches='tight')
plt.show()


## Sección 7 — Conclusiones y relación con MercadoLibre

---

### Conclusión 1 — Brasil y México concentran el mercado, pero el crecimiento está en los medianos

Brasil (70% internet, 214M celulares) y México (72%, 122M) dominan el volumen absoluto de LATAM y son los mercados maduros de MercadoLibre. Sin embargo, el heatmap de YoY muestra que los crecimientos más altos entre 2015 y 2020 ocurren en países de tamaño medio: **Bolivia, Honduras, Nicaragua y Paraguay** superan el 15% YoY de crecimiento en penetración de internet en varios años del período.

**Implicación para MercadoLibre:** los mercados grandes necesitan estrategias de profundización (más SKUs, logística, Mercado Pago). Los mercados medianos en aceleración son la oportunidad de expansión donde la adquisición de nuevos usuarios tiene el costo más bajo y el retorno más alto.

---

### Conclusión 2 — La telefonía móvil supera el 100% de penetración: el e-commerce es mobile-first por necesidad

En 2020, países como Brasil, México, Colombia, Argentina y Chile tienen más suscripciones celulares que población (razón > 1.0). Esto significa que una fracción significativa de la población tiene múltiples SIM cards, y que el celular es el principal o único dispositivo de acceso a internet para millones de usuarios de bajos recursos.

**Implicación para MercadoLibre:** la estrategia mobile-first no es opcional. La experiencia de compra, el onboarding de vendedores y Mercado Pago deben estar optimizados para conexiones lentas, pantallas pequeñas y usuarios con bajo almacenamiento. Los datos justifican inversión en la app móvil por encima del sitio web en todos los mercados de LATAM.

---

### Conclusión 3 — COVID-19 (2020) fue el mayor acelerador de penetración digital en la historia de LATAM

El heatmap muestra que 2020 tiene los valores más altos de YoY de internet en casi todos los países de LATAM, con crecimientos del 7% al 20% en un solo año. Colombia pasó de 65% a 70%, Bolivia de 46% a 53%. Este salto, forzado por el confinamiento, incorporó millones de usuarios digitales que probablemente no habrían adoptado internet por varios años más.

**Implicación para MercadoLibre:** este cohorte de usuarios 'pandémicos' representa tanto una oportunidad como un riesgo. Son usuarios que llegaron por necesidad, no por preferencia digital, y cuya retención requiere educación de producto y simplicidad de experiencia. Las métricas de retención y frecuencia de compra de este cohorte deberían ser un KPI prioritario para 2022–2024.
